In [ ]:
# ============================================================
# MPLUS TIME GRID PIPELINE
# BASELINE EMA + STEPS + HEART RATE
# ============================================================

# TO DO: 
# 1. Add Stress 
# 2. transfer the pipeline from a single notebook in seperate functions (until end of May) 
# and move all in a separate repository

In [ ]:
# ============================================================
# IMPORTS AND SETTINGS
# ============================================================

from pyprojroot import here 
from pathlib import Path
import sys
import pandas as pd
import numpy as np

MISSING = -999

# ============================================================
# PATHS
# ============================================================

# .here is located as invisible file in the project root working directory
PROJECT_ROOT = Path(here())

# add project root to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# import local paths from server_config
from server_config import preprocessed_path, redcap_path

# convert server paths to Path objects and set path
passive_path = Path(preprocessed_path) / "backup_passive_recent.feather"
ema_path = Path(preprocessed_path) / "ema_content.pkl"
redcap_path = Path(redcap_path) / "redcap_data.pkl"

# define output path
OUTPUT_DIR = Path(preprocessed_path) / "tessa" / "dsem"

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df_passive = pd.read_feather(passive_path)
df_ema = pd.read_pickle(ema_path)
df_redcap = pd.read_pickle(redcap_path)


In [ ]:
# display all item names
#sorted(df_ema["item"].dropna().unique())

In [ ]:
# define affect items
na_cols = ['panas_fear1', 
           'panas_fear2', 
           'panas_guilt1',
           'panas_guilt2', 
           'panas_hostility1', 
           'panas_hostility2',
           'panas_sadness1', 
           'panas_sadness2',
           'panas_loneliness']

pa_cols = ['panas_attentiveness', 
           'panas_joviality1', 
           'panas_joviality2',
           'panas_selfassurance',
           'panas_serenity1', 
           'panas_serenity2']

# note: the items ‘panas_shyness’ and ‘panas_fatigue’ have been excluded because they cannot be clearly assigned to either NA or PA

In [ ]:
# ============================================================
# CLEAN EMA
# ============================================================

# 1. keep baseline data only
ema_base = df_ema[df_ema['measurement_burst'] == 0]
# ema_base["measurement_burst"].value_counts() 

# 2. keep relevant columns
df_ema_base = ema_base[['id', 
                        'timestamp_item_completion',
                        'timestamp_beep_completion', 
                        'measurement_burst', 
                        'response', 
                        'item', 
                        'beep_per_person_id',
                        'n_beeps_beeps_completed_per_burst',    # beep count burst (1 - 112)
                        'nr_beep_daily',                        # beep count daily (1 - 8)
                        'n_beeps_completed_per_day',            # sum of beeps completed (per person) within a day
                        'relative_beep_counter'                 # nth beep completed (per person) per burst
                        ]].copy()

# 3. data format 
df_ema_base['id'] = df_ema_base['id'].astype(str)
df_ema_base['response'] = pd.to_numeric(df_ema_base['response'], errors='coerce')

# 4. drop all rows with NaN (prerequisite for pivot)
df_ema_base = df_ema_base.dropna(subset=["id", "beep_per_person_id", "item", "response",])


In [ ]:
# 5. keep affect items only                          TO DO: ADD STRESS ITEMS er_intensity, er_control, event_general
affect_items = pa_cols + na_cols

df_ema_affect_long = df_ema_base[
    df_ema_base["item"].isin(affect_items)
].copy()

df_ema_affect_long.shape

# check item counts
#df_ema_affect_long["item"].value_counts()


In [ ]:
# 6. pivot ema from long to wide format (this creates one row per person beep)
df_ema_wide = (
    df_ema_affect_long
    .pivot_table(
        index=[
            "id",
            "beep_per_person_id",
        ],
        columns="item",
        values="response",
        aggfunc="first"
    )
    .reset_index()
)

df_ema_wide.head()

In [ ]:
# 7. add beep-level timing and metadata
df_beep_info = (
    df_ema_base
    .groupby(["id", "beep_per_person_id"], as_index=False)
    .agg(
        timestamp_beep_completion=("timestamp_beep_completion", "min"),
        timestamp_item_completion=("timestamp_item_completion", "min"),
        measurement_burst=("measurement_burst", "first"),
        n_beeps_completed_per_burst=("n_beeps_beeps_completed_per_burst", "first"),
        nr_beep_daily=("nr_beep_daily", "first"),
        n_beeps_completed_per_day=("n_beeps_completed_per_day", "first"),
        relative_beep_counter=("relative_beep_counter", "first"),
    )
)

df_ema_wide = df_beep_info.merge(
    df_ema_wide,
    on=["id", "beep_per_person_id"],
    how="left"
)

df_ema_wide.head()

In [ ]:
# 8. create PA and NA scores 
df_ema_wide["pa"] = df_ema_wide[pa_cols].mean(axis=1, skipna=True)
df_ema_wide["na"] = df_ema_wide[na_cols].mean(axis=1, skipna=True)

# descriptive stats
df_ema_wide[["pa", "na"]].describe()


In [ ]:
# check missingness
df_ema_wide[["pa", "na"]].isna().mean()

In [ ]:
# ============================================================
# CREATE BASELINE TIME GRID FROM THOSE EMA ROWS
# ============================================================

# rename for readability
df_ema_wide = df_ema_wide.rename(
    columns={
        "timestamp_item_completion": "timestamp"
    }
)

# make sure timestamp is datetime
df_ema_wide["timestamp"] = pd.to_datetime(
    df_ema_wide["timestamp"],
    errors="coerce"
)

# sort by person and time
df_ema_wide = df_ema_wide.sort_values(
    ["id", "timestamp"]
).copy()

# first beep per person
df_ema_wide["t0"] = (df_ema_wide.groupby("id")["timestamp"].transform("first"))

# hours since first beep
df_ema_wide["time_hr"] = ((df_ema_wide["timestamp"] - df_ema_wide["t0"]).dt.total_seconds()/ 3600)

# inspect
df_ema_wide[["id", "timestamp", "time_hr", "pa","na"]].head(10)

# 'time_hr' is the relevant Mplus variable for TINTERVAL = time (2.5)
# if you only want to create a time grid for ema data you can stop here and export it!


In [ ]:
# no missings - CHECK :)
rows_missing_time = df_ema_wide[
    df_ema_wide["timestamp"].isna()
].copy()

rows_missing_time.head()

In [ ]:
# create passive-data windows
# block_end = current EMA timestamp
df_ema_wide["block_end"] = df_ema_wide["timestamp"]

# block_start = previous EMA timestamp
df_ema_wide["block_start"] = (
    df_ema_wide
    .groupby("id")["timestamp"]
    .shift(1)
)

# for first beep, use 2.5 hours before first EMA
df_ema_wide["block_start"] = df_ema_wide["block_start"].fillna(
    df_ema_wide["block_end"] - pd.Timedelta(hours=2.5)
)

# unique block ID for mapping passive data
df_ema_wide["unique_block"] = (
    df_ema_wide["id"].astype(str)
    + "_"
    + df_ema_wide.groupby("id").cumcount().add(1).astype(str)
)

# create grid object for later mapping
df_grid = df_ema_wide[
    [
        "id",
        "unique_block",
        "timestamp",
        "block_start",
        "block_end",
        "time_hr",
        "pa",
        "na",
    ]
].copy()

# inspect
df_grid.head(10)

This time grid is ready for both:
1. Mplus EMA-only .dat export
2. steps / heart rate mapping

In [ ]:
df_passive.head()

In [ ]:
# ============================================================
# CLEAN STEPS 
# ============================================================

df_steps = df_passive[df_passive["modality"] == "Steps"].copy()

df_steps["id"] = df_steps["id"].astype(str)

df_steps["timestamp_start"] = pd.to_datetime(
    df_steps["timestamp_start"],
    errors="coerce"
)

df_steps["timestamp_end"] = pd.to_datetime(
    df_steps["timestamp_end"],
    errors="coerce"
)

df_steps["float_value"] = pd.to_numeric(
    df_steps["float_value"],
    errors="coerce"
)

df_steps = df_steps.dropna(
    subset=["id", "timestamp_start", "timestamp_end", "float_value"]
)

df_steps = df_steps[df_steps["float_value"] >= 0]

df_steps["duration_min"] = (
    df_steps["timestamp_end"] - df_steps["timestamp_start"]
).dt.total_seconds() / 60

df_steps = df_steps[df_steps["duration_min"] > 0]

df_steps["steps_per_min"] = df_steps["float_value"] / df_steps["duration_min"]

# in line with Hammelrath et al. (2026): 200 steps per minute as threshold 
df_steps = df_steps[df_steps["steps_per_min"] <= 200] 

df_steps = df_steps.drop(columns=["duration_min", "steps_per_min"])

df_steps.head(100)


In [ ]:
# ============================================================
# CLEAN HR 
# ============================================================

df_hr = df_passive[df_passive["modality"] == "HeartRate"].copy()

df_hr["id"] = df_hr["id"].astype(str)

df_hr["timestamp_start"] = pd.to_datetime(
    df_hr["timestamp_start"],
    errors="coerce"
)

df_hr["float_value"] = pd.to_numeric(
    df_hr["float_value"],
    errors="coerce"
)

df_hr = df_hr.dropna(
    subset=["id", "timestamp_start", "float_value"]
)

# in line with Hammelrath et al. (2026) p. 3
df_hr = df_hr[df_hr["float_value"].between(30, 220)]

df_hr.head()

In [ ]:
# ============================================================
# RESTRICT PASSIVE DATA TO BASELINE
# ============================================================

baseline_period = (
    df_grid
    .groupby("id")
    .agg(
        baseline_start=("block_start", "min"),
        baseline_end=("block_end", "max")
    )
    .reset_index()
)

baseline_period.head()

In [ ]:
# STEPS
df_steps = df_steps.merge(
    baseline_period,
    on="id",
    how="inner"
)

df_steps = df_steps[
    (df_steps["timestamp_start"] <= df_steps["baseline_end"]) &
    (df_steps["timestamp_end"] >= df_steps["baseline_start"])
].copy()

df_steps = df_steps.drop(columns=["baseline_start", "baseline_end"])

df_steps.shape

In [ ]:
# HEART RATE 
df_hr = df_hr.merge(
    baseline_period,
    on="id",
    how="inner"
)

df_hr = df_hr[
    (df_hr["timestamp_start"] >= df_hr["baseline_start"]) &
    (df_hr["timestamp_start"] <= df_hr["baseline_end"])
].copy()

df_hr = df_hr.drop(columns=["baseline_start", "baseline_end"])

df_hr.shape


In [ ]:
# ============================================================
# MAP PASSIVE DATA INTO BASELINE GRID
# ============================================================

# map steps to ema blocks
joined_steps = df_steps.merge(
    df_grid[
        [
            "id",
            "unique_block",
            "block_start",
            "block_end"
        ]
    ],
    on="id",
    how="inner"
)

joined_steps = joined_steps[
    (joined_steps["timestamp_start"] < joined_steps["block_end"]) &
    (joined_steps["timestamp_end"] > joined_steps["block_start"])
].copy()

joined_steps.shape

In [ ]:
# calculate weighted steps

# For each step record and EMA block, take the later start time 
# (the overlap cannot start before the EMA block starts)
joined_steps["overlap_start"] = joined_steps[
    ["timestamp_start", "block_start"]
].max(axis=1)

# For each step record and EMA block, take the earlier end time 
# (the overlap cannot continue after the step record ends)
joined_steps["overlap_end"] = joined_steps[
    ["timestamp_end", "block_end"]
].min(axis=1)

# calculate how many seconds the step record overlaps with the EMA block
joined_steps["overlap_seconds"] = (
    joined_steps["overlap_end"] - joined_steps["overlap_start"]
).dt.total_seconds()

# calculate the total duration of the original step record
joined_steps["sensor_seconds"] = (
    joined_steps["timestamp_end"] - joined_steps["timestamp_start"]
).dt.total_seconds()

# remove invalid step records where the duration is zero or negative
joined_steps = joined_steps[joined_steps["sensor_seconds"] > 0]

# assign only the proportional amount of steps that belongs to the EMA block
# example: overlap_seconds = 1800, sensor_seconds  = 3600, float_value = 600
# weighted_steps = 1800 / 3600 × 600 = 300 -> only 300 of the 600 steps are assigned to this EMA block
joined_steps["weighted_steps"] = (
    joined_steps["overlap_seconds"] / joined_steps["sensor_seconds"]
) * joined_steps["float_value"]

# sum all weighted step records within each EMA block
steps_summary = (
    joined_steps
    .groupby("unique_block")["weighted_steps"]
    .sum()
    .reset_index(name="steps")
)

steps_summary.head()

In [ ]:
# merge onto grid
df_grid = df_grid.merge(
    steps_summary,
    on="unique_block",
    how="left"
)

df_grid.head()

In [ ]:
# steps are highly skewed
df_grid["steps"].hist(bins=50)

# if it looks strongly skewed use log transformation to reduce skewness and stabilize estimation
#np.log1p()

#raw steps: absolute physical activity volume
#log-transformed steps: relative/intensity-scaled activity variation 

In [ ]:
# map heart rate to ema blocks
joined_hr = df_hr.merge(
    df_grid[
        [
            "id",
            "unique_block",
            "block_start",
            "block_end"
        ]
    ],
    on="id",
    how="inner"
)

joined_hr = joined_hr[
    (joined_hr["timestamp_start"] >= joined_hr["block_start"]) &
    (joined_hr["timestamp_start"] < joined_hr["block_end"])
].copy()

joined_hr.shape


In [ ]:
# aggregate HR
hr_summary = (
    joined_hr
    .groupby("unique_block")["float_value"]
    .agg(
        hr_mean="mean",
        hr_median="median",
        hr_sd="std",   
        hr_n="count"
    )
    .reset_index()
)

hr_summary.head()

In [ ]:
# merge into grid
df_grid = df_grid.merge(
    hr_summary,
    on="unique_block",
    how="left"
)

df_grid.head()

In [ ]:
# check final missingness
df_grid[["pa", "na", "steps", "hr_mean"]].isna().mean()

In [ ]:
# check distributions
df_grid[["pa", "na", "steps", "hr_mean"]].describe()

In [ ]:
# optional: log-transform steps (recommended for modeling?) 
df_grid["steps_log"] = np.log1p(df_grid["steps"])

df_grid[["steps", "steps_log"]].describe()

df_grid.head()

In [ ]:
# ============================================================
# CREATE FINAL MPLUS DATASET
# ============================================================

mplus_affect_steps_hr = df_grid[
    [
        "id",
        "time_hr",
        "pa",
        "na",
        "steps",
        "steps_log",
        "hr_mean",
        "hr_median",
        "hr_sd"
    ]
].copy()


# replace infinite values
mplus_affect_steps_hr = mplus_affect_steps_hr.replace(
    [np.inf, -np.inf],
    np.nan
)

# replace missing values for Mplus (cannot handle NaN, only -999)
mplus_affect_steps_hr = mplus_affect_steps_hr.fillna(MISSING)

# important for later modeling: ensure appropriate sorting (in ascending order) -> observations must be ordered
mplus_affect_steps_hr = (mplus_affect_steps_hr.sort_values(["id", "time_hr"]).reset_index(drop=True))

# sanity check
mplus_affect_steps_hr.head(20)

In [ ]:
# save .csv file (with headers) / good for inspection
mplus_affect_steps_hr.to_csv(
    OUTPUT_DIR / "mplus_affect_steps_hr.csv",
    index=False
)

# save .dat file for Mplus (tab-delimited, header-less: suitable for Mplus)
mplus_affect_steps_hr.to_csv(
    OUTPUT_DIR / "mplus_affect_steps_hr.dat",
    sep=" ",
    header=False,
    index=False
)

In [ ]:
# example reponse distribution
#df_ema_wide['panas_attentiveness'].value_counts().sort_index()